# 34. DPO — 선호로 학습하기

> **제34장** · **이론편 대응: 23장 (정렬과 추론)**
> **예상 소요**: 80분 (학습 3~5분)
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: 없음 (31~32장의 transformers, peft 사용)
> **다운로드**: DistilGPT-2 (31장에서 받았다면 재사용)

---

## 이 장에서 하는 일

31장의 SFT는 **"이렇게 답하라"**를 가르쳤다. 정렬은 다르다.

> **"이 답이 저 답보다 낫다"**를 가르친다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | SFT만으로 부족한 이유 | 23.1절 |
| 2 | RLHF와 DPO의 차이 | 23.2~23.3절 |
| 3 | **DPO 손실 0.644 검증** ★ | 23.3절 |
| 4 | 선호 데이터 만들기 | 23.3절 |
| 5 | 로그 확률 계산 | 23.3절 |
| 6 | DPO 학습 실행 | 23.3절 |
| 7 | 학습 전후 비교 | 23.3절 |
| 8 | 정렬의 위험 | 23.5절 |

**3절이 핵심이다.** 이론편 23.3절에서 손으로 계산한 세 값을 확인하고,
그 식이 무엇을 유도하는지 이해한다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
import math
import copy
import time

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} / 장치: {device}")

from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
print(f"모델: {MODEL_NAME}")

---

## 1. SFT만으로 부족한 이유 — 이론편 23.1절

31장에서 SFT로 "질문에 답하는 형식"을 가르쳤다. 그런데 문제가 남는다.

**정답이 하나가 아닌 경우**가 많기 때문이다.

| 질문 | 답변 A | 답변 B |
|---|---|---|
| "이 코드 어때?" | "형편없네요. 다시 쓰세요." | "동작은 하지만 개선할 점이 있습니다." |
| "다이어트 약 추천해줘" | (특정 약품명 나열) | "전문의 상담을 권합니다." |

**둘 다 문법적으로 옳고 질문에 답하고 있다.** SFT는 "어느 쪽이 더 나은가"를 가르치지 못한다.

이론편 23.1절에서 다룬 정렬(alignment)이 이 문제를 다룬다.
**사람이 선호하는 방향으로 모델을 맞추는 것**이다.

In [ ]:
print("=" * 70)
print("SFT와 정렬의 차이 (이론편 23.1절)")
print("=" * 70)
print()
print(f"{'구분':<14}{'학습 데이터':<28}{'가르치는 것'}")
print("-" * 70)
print(f"{'SFT':<14}{'(질문, 모범답변)':<28}{'이렇게 답하라'}")
print(f"{'정렬':<14}{'(질문, 좋은답, 나쁜답)':<28}{'이 답이 저 답보다 낫다'}")
print("-" * 70)
print()
print("데이터 형태가 다르다는 점에 주목하자.")
print("  SFT : 정답 하나")
print("  정렬 : 두 답변의 **순위**")
print()
print("왜 순위가 더 쉬운가")
print("  '완벽한 답'을 쓰는 것은 어렵다.")
print("  '둘 중 어느 쪽이 나은가'를 고르는 것은 훨씬 쉽다.")
print()
print("  → 사람이 데이터를 만들기 쉬워진다 (이론편 23.2절)")

---

## 2. RLHF와 DPO — 이론편 23.2~23.3절

정렬 방법에는 두 갈래가 있다.

### RLHF (이론편 23.2절)

1. 사람의 선호 데이터로 **보상 모델**을 학습
2. 그 보상을 최대화하도록 **강화학습**(17장)으로 정책 학습

**모델이 세 개 필요하다** — 정책, 보상 모델, 참조 모델. 학습이 불안정하고 복잡하다.

### DPO (이론편 23.3절)

**보상 모델 없이 바로 학습한다.** 수학적으로 RLHF와 같은 목표를 최적화한다는 것이 논문의 요지다.

$$L_{DPO} = -\log\sigma\!\left(\beta\left[\log\frac{\pi(y_w|x)}{\pi_{ref}(y_w|x)} - \log\frac{\pi(y_l|x)}{\pi_{ref}(y_l|x)}\right]\right)$$

복잡해 보이지만 **하는 일은 단순하다** — 3절에서 확인한다.

In [ ]:
print("=" * 78)
print("RLHF vs DPO (이론편 23.2~23.3절)")
print("=" * 78)
print()
print(f"{'항목':<20}{'RLHF':<28}{'DPO'}")
print("-" * 78)
rows = [
    ("필요한 모델", "정책 + 보상 + 참조 (3개)", "정책 + 참조 (2개)"),
    ("학습 방식", "강화학습 (PPO 등)", "지도학습과 유사"),
    ("보상 모델", "따로 학습 필요", "불필요"),
    ("안정성", "불안정 — 조정이 까다로움", "비교적 안정"),
    ("구현 난이도", "높음", "낮음"),
    ("메모리", "많이 필요", "상대적으로 적음"),
]
for a, b, c in rows:
    print(f"{a:<20}{b:<28}{c}")
print("-" * 78)
print()
print("DPO가 널리 쓰이는 이유는 **구현이 단순하면서 효과가 비슷하기 때문**이다.")
print()
print("다만 참조 모델은 여전히 필요하다.")
print("  '원래 모델에서 얼마나 벗어났는가'를 재기 위해서다 (3절).")

---

## 3. DPO 손실 검증 — 이론편 23.3절 값 ★

식이 복잡해 보이므로 **괄호 안의 값을 $\Delta$로 줄여** 보자.

$$L = -\log\sigma(\beta\Delta), \qquad \Delta = \underbrace{\log\frac{\pi(y_w)}{\pi_{ref}(y_w)}}_{\text{선호 응답이 오른 정도}} - \underbrace{\log\frac{\pi(y_l)}{\pi_{ref}(y_l)}}_{\text{비선호가 오른 정도}}$$

**이론편 23.3절에서 계산한 세 상황** ($\beta = 0.1$)

| 학습 상황 | $\Delta$ | $\sigma(\beta\Delta)$ | 손실 |
|---|---|---|---|
| 선호 응답만 확률 상승 | +1.0 | 0.525 | **0.644** |
| 둘 다 똑같이 상승 | 0.0 | 0.500 | 0.693 |
| 비선호만 상승 | −1.0 | 0.475 | **0.744** |

In [ ]:
import math
import numpy as np

BETA = 0.1

def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))

print("=" * 78)
print("이론편 23.3절 값 검증")
print("=" * 78)
print(f"beta = {BETA}")
print()
print(f"{'학습 상황':<24}{'Δ':<10}{'βΔ':<10}{'σ(βΔ)':<12}{'손실':<12}{'이론편'}")
print("-" * 78)

cases = [
    ("선호 응답만 확률 상승",  1.0, 0.644),
    ("둘 다 똑같이 상승",     0.0, 0.693),
    ("비선호만 상승",        -1.0, 0.744),
]

for name, delta, book in cases:
    bd = BETA * delta
    s = sigmoid(bd)
    loss = -math.log(s)
    print(f"{name:<24}{delta:<10.1f}{bd:<10.2f}{s:<12.4f}{loss:<12.4f}{book:.3f}")
    assert abs(loss - book) < 0.001, f"{name}이 이론편 값과 다릅니다"

print("-" * 78)
print("[OK] 이론편 23.3절 손계산과 일치")

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print("이 손실이 유도하는 방향")
print("=" * 70)
print()
print("세 값을 비교하면")
print(f"  선호만 상승 : 0.644  ← 가장 낮음")
print(f"  둘 다 상승  : 0.693")
print(f"  비선호 상승 : 0.744  ← 가장 높음")
print()
print("학습은 손실을 줄이는 쪽으로 진행되므로, 자연히 첫 번째 방향으로 밀려간다.")
print()
print("[핵심] 둘째 행이 특히 중요하다")
print("  선호·비선호를 똑같이 올리면 Δ=0 이 되어 손실이 개선되지 않는다.")
print("  즉 DPO는 '전반적으로 다 잘하게' 만드는 것이 아니라")
print("  **둘 사이의 격차를 벌리도록** 학습시킨다.")
print()
print("  시험에서 절대 점수가 아니라 등수 차이로 평가하는 것과 비슷하다.")

# 그래프
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
deltas = np.linspace(-5, 5, 200)
for beta, color in [(0.1, "#1E40AF"), (0.5, "#EA580C"), (1.0, "#0D9488")]:
    losses = [-math.log(sigmoid(beta * d)) for d in deltas]
    ax.plot(deltas, losses, linewidth=2, color=color, label=f"β={beta}")
ax.axvline(0, color="gray", linestyle="--", linewidth=1)
ax.axhline(math.log(2), color="gray", linestyle=":", linewidth=1)
ax.text(3, math.log(2)+0.05, "ln2 = 0.693", fontsize=8, color="gray")
ax.set_xlabel("Δ (선호 우위)")
ax.set_ylabel("DPO 손실")
ax.set_title("Δ에 따른 손실")
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
names = [c[0] for c in cases]
vals = [-math.log(sigmoid(BETA * c[1])) for c in cases]
colors = ["#0D9488", "#94A3B8", "#DC2626"]
bars = ax.bar(range(3), vals, color=colors)
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.005, f"{v:.4f}", ha="center", fontsize=10)
ax.set_xticks(range(3))
ax.set_xticklabels(["선호만\n상승", "둘 다\n상승", "비선호\n상승"], fontsize=9)
ax.set_ylabel("손실")
ax.set_title("세 상황의 손실 (β=0.1)")
ax.set_ylim(0.6, 0.78)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn.functional as F

print("=" * 70)
print("PyTorch로 구현")
print("=" * 70)


def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    """DPO 손실 (이론편 23.3절)

    각 인자는 로그 확률의 합이다.
    logsigmoid 를 쓰는 이유: -log(sigmoid(x)) 를 안정적으로 계산하기 위해
    """
    # 정책이 참조 대비 얼마나 올랐는가
    chosen_reward = policy_chosen_logps - ref_chosen_logps
    rejected_reward = policy_rejected_logps - ref_rejected_logps

    delta = chosen_reward - rejected_reward
    loss = -F.logsigmoid(beta * delta)

    return loss, chosen_reward, rejected_reward


print("세 상황을 텐서로 계산")
print(f"{'상황':<20}{'손실':<14}{'이론편'}")
print("-" * 70)
for name, (pw, pl, rw, rl), book in [
    ("선호만 상승", (1.0, 0.0, 0.0, 0.0), 0.644),
    ("둘 다 상승",  (1.0, 1.0, 0.0, 0.0), 0.693),
    ("비선호 상승", (0.0, 1.0, 0.0, 0.0), 0.744),
]:
    t = lambda v: torch.tensor(v)
    loss, _, _ = dpo_loss(t(pw), t(pl), t(rw), t(rl), beta=0.1)
    print(f"{name:<20}{loss.item():<14.4f}{book:.3f}")
    assert abs(loss.item() - book) < 0.001

print("-" * 70)
print("[OK] 앞서 손으로 계산한 값과 일치")
print()
print("F.logsigmoid 를 쓰는 이유")
print("  -log(sigmoid(x)) 를 직접 계산하면 x가 크게 음수일 때 오버플로가 난다.")
print("  logsigmoid 는 수치적으로 안정된 방식으로 계산한다.")
print("  20장 4절의 소프트맥스 최댓값 빼기와 같은 종류의 대응이다.")

### 참조 모델이 왜 필요한가

식에 $\pi_{ref}$가 들어 있다. **학습 시작 시점의 모델을 그대로 복사해 얼려 둔 것**이다.

없으면 어떻게 될까. 모델이 선호 응답의 확률만 무한정 올리려 할 것이고,
결국 **원래 능력을 잃어버린다**(catastrophic forgetting).

참조 모델과의 비율을 보면 **"원래보다 얼마나 올랐는가"**를 재게 되므로,
원본에서 지나치게 벗어나는 것을 억제한다.

---

## 4. 선호 데이터 만들기 — 이론편 23.3절

DPO 학습 데이터는 **세 가지**로 이루어진다.

```python
{
    "prompt":   "질문",
    "chosen":   "선호하는 응답",
    "rejected": "선호하지 않는 응답",
}
```

이 장에서는 **간결하고 정중한 답변을 선호**하도록 가르쳐 본다.

In [ ]:
# 선호 데이터 — chosen은 간결하고 정중, rejected는 장황하거나 무례
preference_data = [
    {
        "prompt": "Q: What is Python?\nA:",
        "chosen": " Python is a programming language known for readable syntax.",
        "rejected": " Well, um, Python is like, you know, a thing people use for stuff.",
    },
    {
        "prompt": "Q: How do I fix this bug?\nA:",
        "chosen": " Check the error message first, then verify your input types.",
        "rejected": " Your code is terrible. Just rewrite everything from scratch.",
    },
    {
        "prompt": "Q: What is a variable?\nA:",
        "chosen": " A variable is a named container that holds a value.",
        "rejected": " A variable is a variable that varies and is variable.",
    },
    {
        "prompt": "Q: Should I learn Python or Java?\nA:",
        "chosen": " Both are useful. Python is easier for beginners to start with.",
        "rejected": " Java is garbage. Only idiots use Java.",
    },
    {
        "prompt": "Q: What is machine learning?\nA:",
        "chosen": " Machine learning finds patterns in data to make predictions.",
        "rejected": " Machine learning is when machines learn things by learning.",
    },
    {
        "prompt": "Q: Is my code good?\nA:",
        "chosen": " It works, but adding comments would improve readability.",
        "rejected": " No. It is bad. Very bad. I cannot even look at it.",
    },
]

print("=" * 78)
print("선호 데이터")
print("=" * 78)
print(f"총 {len(preference_data)}쌍")
print()

for i, ex in enumerate(preference_data[:3], 1):
    print(f"[{i}] {ex['prompt'].strip()}")
    print(f"    선호   : {ex['chosen'].strip()}")
    print(f"    비선호 : {ex['rejected'].strip()}")
    print()

print("-" * 78)
print("이 데이터가 가르치려는 것")
print("  - 무례하지 않게 답하기")
print("  - 동어반복 피하기")
print("  - 구체적으로 답하기")
print()
print("모두 '무엇이 정답인가'가 아니라 '어떻게 답하는가'의 문제다.")

---

## 5. 로그 확률 계산 — 이론편 23.3절

DPO 손실에는 $\log\pi(y|x)$가 들어간다. **응답 전체의 로그 확률**이다.

$$\log\pi(y|x) = \sum_{t}\log P(y_t \mid x, y_{<t})$$

31장에서 만든 손실 마스킹과 관계가 있다. **응답 부분의 토큰만 더한다.**

In [ ]:
import torch


def get_logprobs(model, prompt, response, device=device):
    """응답 부분의 로그 확률 합을 구한다 (이론편 23.3절)

    31장 3절의 손실 마스킹과 같은 방식으로 프롬프트를 제외한다.
    """
    prompt_ids = tokenizer.encode(prompt)
    response_ids = tokenizer.encode(response)

    input_ids = torch.tensor([prompt_ids + response_ids]).to(device)
    labels = torch.tensor([[-100] * len(prompt_ids) + response_ids]).to(device)

    outputs = model(input_ids=input_ids)

    # 다음 토큰 예측이므로 한 칸씩 밀어서 맞춘다
    logits = outputs.logits[:, :-1]      # 마지막 예측은 쓰지 않음
    targets = labels[:, 1:]              # 첫 토큰은 예측 대상이 아님

    log_probs = torch.log_softmax(logits, dim=-1)

    # 각 위치에서 실제 토큰의 로그 확률을 뽑는다
    mask = (targets != -100)
    safe_targets = targets.clamp(min=0)
    token_logps = torch.gather(
        log_probs, 2, safe_targets.unsqueeze(-1)).squeeze(-1)

    # 마스크된 부분(프롬프트)은 제외하고 합산
    return (token_logps * mask).sum()


model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

ex = preference_data[0]

print("=" * 70)
print("로그 확률 계산")
print("=" * 70)
print(f"프롬프트: {ex['prompt'].strip()}")
print()

with torch.no_grad():
    lp_chosen = get_logprobs(model, ex["prompt"], ex["chosen"])
    lp_rejected = get_logprobs(model, ex["prompt"], ex["rejected"])

print(f"{'응답':<12}{'로그 확률':<16}{'내용'}")
print("-" * 70)
print(f"{'선호':<12}{lp_chosen.item():<16.4f}{ex['chosen'][:36]}...")
print(f"{'비선호':<12}{lp_rejected.item():<16.4f}{ex['rejected'][:36]}...")
print("-" * 70)
print()
print("로그 확률은 항상 음수다 (확률이 0~1 이므로 log는 음수).")
print("값이 클수록(0에 가까울수록) 모델이 그 응답을 만들 가능성이 높다는 뜻이다.")
print()
print("주의: 응답이 길수록 더 많은 토큰이 더해져 값이 작아진다.")
print("  길이가 크게 다르면 비교가 왜곡될 수 있어, 평균을 쓰기도 한다.")

---

## 6. DPO 학습 실행 — 이론편 23.3절

준비가 끝났다. **참조 모델을 복사해 얼리고** 학습을 시작한다.

In [ ]:
import torch
import copy

print("=" * 70)
print("정책 모델과 참조 모델")
print("=" * 70)

# 정책 모델 — 학습할 대상
policy_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

# 참조 모델 — 학습 시작 시점을 그대로 보존
ref_model = copy.deepcopy(policy_model)
for p in ref_model.parameters():
    p.requires_grad = False
ref_model.eval()

n_policy = sum(p.numel() for p in policy_model.parameters() if p.requires_grad)
n_ref = sum(p.numel() for p in ref_model.parameters() if p.requires_grad)

print(f"정책 모델 학습 대상: {n_policy:,}개")
print(f"참조 모델 학습 대상: {n_ref:,}개  (전부 얼림)")
print()
print("메모리 사용")
size = sum(p.numel() * p.element_size() for p in policy_model.parameters())
print(f"  정책 모델: {size/1024**2:.0f} MB")
print(f"  참조 모델: {size/1024**2:.0f} MB")
print(f"  합계     : {size*2/1024**2:.0f} MB")
print()
print("참조 모델 때문에 메모리가 두 배로 든다.")
print("  32장의 LoRA를 함께 쓰면 이 부담을 줄일 수 있다 —")
print("  원본 가중치를 공유하고 어댑터만 다르게 두는 방식이다.")

In [ ]:
import torch
import time
import numpy as np

print("=" * 70)
print("DPO 학습")
print("=" * 70)

BETA = 0.1
EPOCHS = 12
LR = 5e-6           # SFT보다 훨씬 작게 — 원본에서 크게 벗어나지 않도록

optimizer = torch.optim.AdamW(policy_model.parameters(), lr=LR)

history = {"loss": [], "chosen_reward": [], "rejected_reward": [], "margin": []}
t0 = time.time()

for epoch in range(EPOCHS):
    policy_model.train()
    epoch_stats = {"loss": [], "cr": [], "rr": []}

    for ex in preference_data:
        # 정책 모델의 로그 확률
        pol_chosen = get_logprobs(policy_model, ex["prompt"], ex["chosen"])
        pol_rejected = get_logprobs(policy_model, ex["prompt"], ex["rejected"])

        # 참조 모델의 로그 확률 (그래디언트 불필요)
        with torch.no_grad():
            ref_chosen = get_logprobs(ref_model, ex["prompt"], ex["chosen"])
            ref_rejected = get_logprobs(ref_model, ex["prompt"], ex["rejected"])

        loss, cr, rr = dpo_loss(pol_chosen, pol_rejected,
                                ref_chosen, ref_rejected, beta=BETA)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_stats["loss"].append(loss.item())
        epoch_stats["cr"].append(cr.item())
        epoch_stats["rr"].append(rr.item())

    history["loss"].append(np.mean(epoch_stats["loss"]))
    history["chosen_reward"].append(np.mean(epoch_stats["cr"]))
    history["rejected_reward"].append(np.mean(epoch_stats["rr"]))
    history["margin"].append(history["chosen_reward"][-1] - history["rejected_reward"][-1])

    if (epoch + 1) % 3 == 0:
        print(f"  에폭 {epoch+1:2}: 손실 {history['loss'][-1]:.4f}  "
              f"선호 {history['chosen_reward'][-1]:+.3f}  "
              f"비선호 {history['rejected_reward'][-1]:+.3f}  "
              f"격차 {history['margin'][-1]:+.3f}  ({time.time()-t0:.0f}초)")

print("-" * 70)
print(f"소요 시간: {time.time()-t0:.0f}초")
print(f"초기 손실: {history['loss'][0]:.4f}  (이론값 ln2 = {np.log(2):.4f})")
print(f"최종 손실: {history['loss'][-1]:.4f}")

### 초기 손실이 ln2인 이유

학습 시작 시점에는 **정책 모델과 참조 모델이 같다.** 따라서

$$\log\frac{\pi(y_w)}{\pi_{ref}(y_w)} = \log 1 = 0, \qquad \log\frac{\pi(y_l)}{\pi_{ref}(y_l)} = 0$$

이므로 $\Delta = 0$이고, 손실은 $-\log\sigma(0) = -\log 0.5 = \log 2 \approx 0.693$이 된다.

**3절의 "둘 다 똑같이 상승"과 같은 상태**다. 학습이 진행되며 격차가 벌어져야 손실이 준다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

epochs = range(1, len(history["loss"]) + 1)

# --- 손실 ---
ax = axes[0]
ax.plot(epochs, history["loss"], marker="o", linewidth=2, color="#1E40AF")
ax.axhline(np.log(2), color="gray", linestyle="--", linewidth=1.5)
ax.text(len(epochs)*0.55, np.log(2)+0.004, "ln2 (시작점)", fontsize=8, color="gray")
ax.set_xlabel("에폭"); ax.set_ylabel("DPO 손실")
ax.set_title("손실")
ax.grid(alpha=0.3)

# --- 보상 (참조 대비 변화) ---
ax = axes[1]
ax.plot(epochs, history["chosen_reward"], marker="o", linewidth=2,
        label="선호 응답", color="#0D9488")
ax.plot(epochs, history["rejected_reward"], marker="s", linewidth=2,
        label="비선호 응답", color="#DC2626")
ax.axhline(0, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("에폭"); ax.set_ylabel("참조 대비 변화")
ax.set_title("각 응답의 확률 변화")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# --- 격차 ---
ax = axes[2]
ax.plot(epochs, history["margin"], marker="o", linewidth=2, color="#EA580C")
ax.axhline(0, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("에폭"); ax.set_ylabel("격차 (선호 - 비선호)")
ax.set_title("선호 우위 (Δ)")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("가운데 그래프가 DPO의 동작을 잘 보여준다.")
print(f"  선호 응답  : {history['chosen_reward'][0]:+.3f} → {history['chosen_reward'][-1]:+.3f}")
print(f"  비선호 응답: {history['rejected_reward'][0]:+.3f} → {history['rejected_reward'][-1]:+.3f}")
print()
print("선호는 올리고 비선호는 내린다 — 3절에서 본 손실이 유도한 방향이다.")

---

## 7. 학습 전후 비교 — 이론편 23.3절

**실제로 답변이 달라졌는지** 확인한다.

In [ ]:
import torch

print("=" * 78)
print("선호/비선호 응답의 확률 변화")
print("=" * 78)

policy_model.eval()

print(f"{'질문':<26}{'선호 변화':<14}{'비선호 변화':<14}{'판정'}")
print("-" * 78)

improved = 0
for ex in preference_data:
    with torch.no_grad():
        pol_c = get_logprobs(policy_model, ex["prompt"], ex["chosen"])
        pol_r = get_logprobs(policy_model, ex["prompt"], ex["rejected"])
        ref_c = get_logprobs(ref_model, ex["prompt"], ex["chosen"])
        ref_r = get_logprobs(ref_model, ex["prompt"], ex["rejected"])

    dc = (pol_c - ref_c).item()
    dr = (pol_r - ref_r).item()
    ok = dc > dr
    improved += ok

    q = ex["prompt"].replace("Q: ", "").replace("\nA:", "")
    print(f"{q[:24]:<26}{dc:+14.4f}{dr:+14.4f}{'개선' if ok else '미개선':>8}")

print("-" * 78)
print(f"개선된 항목: {improved}/{len(preference_data)}")
print()
print("선호 응답의 변화량이 비선호보다 크면 학습이 의도대로 된 것이다.")

In [ ]:
import torch

def generate(model, prompt, max_new_tokens=25):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()


print("=" * 78)
print("실제 생성 비교")
print("=" * 78)

test_prompts = [
    "Q: What is Python?\nA:",
    "Q: Is my code good?\nA:",
]

for prompt in test_prompts:
    print(f"\n{prompt.strip()}")
    print(f"  학습 전: {generate(ref_model, prompt)[:66]}")
    print(f"  학습 후: {generate(policy_model, prompt)[:66]}")

print()
print("=" * 78)
print("주의해서 볼 것")
print()
print("데이터가 6쌍뿐이고 학습률도 작게 잡았으므로 변화가 미미할 수 있다.")
print("실무에서는 수천~수만 쌍의 선호 데이터를 쓴다.")
print()
print("이 실습의 목적은 '큰 변화를 보는 것'이 아니라")
print("**DPO가 어떤 방향으로 모델을 미는지** 확인하는 것이다.")
print("6절의 그래프에서 그 방향이 명확히 보였다.")

---

## 8. 정렬의 위험 — 이론편 23.5절

이론편 23.5절에서 정렬 학습의 부작용을 다뤘다. 실무에서 실제로 관찰되는 것들이다.

In [ ]:
print("=" * 78)
print("정렬 학습의 부작용 (이론편 23.5절)")
print("=" * 78)
print()
print(f"{'현상':<20}{'설명':<34}{'대응'}")
print("-" * 78)
issues = [
    ("과도한 회피", "안전한 질문에도 답을 거부",   "거부 사례를 데이터에 포함"),
    ("아부 (sycophancy)", "사용자 의견에 무조건 동조", "반대 의견도 선호로 표시"),
    ("장황해짐", "긴 답변이 선호되면 늘어짐",     "길이를 보정하거나 평균 사용"),
    ("능력 저하", "원래 잘하던 것을 못하게 됨",   "beta 조정, 참조 모델 활용"),
    ("선호 편향", "데이터 작성자의 취향이 반영",   "다양한 평가자 확보"),
]
for a, b, c in issues:
    print(f"{a:<20}{b:<34}{c}")
print("-" * 78)
print()
print("[중요] '정렬'은 가치 판단을 포함한다.")
print("  누구의 선호에 맞출 것인가는 기술적 문제가 아니라 사회적 문제다.")
print("  이론편 23.5절에서 다룬 논쟁이 여기에 있다.")

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print("beta 값의 역할")
print("=" * 70)
print()
print("beta 는 '참조 모델에서 얼마나 벗어나도 되는가'를 조절한다.")
print()
print(f"{'beta':<10}{'Δ=1일 때 손실':<20}{'성격'}")
print("-" * 70)
for beta in [0.01, 0.1, 0.5, 1.0]:
    loss = -math.log(1/(1+math.exp(-beta*1.0)))
    if beta < 0.05:
        note = "매우 보수적 — 거의 안 바뀜"
    elif beta < 0.3:
        note = "일반적으로 쓰는 범위"
    else:
        note = "공격적 — 원본에서 크게 벗어남"
    print(f"{beta:<10}{loss:<20.4f}{note}")
print("-" * 70)
print()
print("beta 가 크면")
print("  선호를 강하게 반영하지만, 원래 능력을 잃을 위험이 커진다.")
print()
print("beta 가 작으면")
print("  안전하지만 변화가 거의 없다.")
print()
print("실무에서는 0.1 ~ 0.5 사이를 많이 쓰며, 실험으로 정한다.")

fig, ax = plt.subplots(figsize=(8, 4.5))
deltas = np.linspace(-3, 3, 200)
for beta, color in [(0.01, "#94A3B8"), (0.1, "#1E40AF"),
                    (0.5, "#EA580C"), (1.0, "#DC2626")]:
    losses = [-math.log(1/(1+math.exp(-beta*d))) for d in deltas]
    ax.plot(deltas, losses, linewidth=2, color=color, label=f"β={beta}")
ax.axvline(0, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Δ (선호 우위)")
ax.set_ylabel("손실")
ax.set_title("beta에 따른 손실 곡선의 기울기")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print()
print("beta 가 클수록 곡선이 가파르다 = 같은 Δ 변화에 손실이 크게 반응한다.")
print("  → 학습이 더 공격적으로 진행된다.")

---

## 9. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| **23.3** | **DPO 손실 0.644 / 0.693 / 0.744** | **일치** ✓ |
| 23.3 | 초기 손실 = ln2 | 검증 ✓ |
| 23.3 | 선호↑ 비선호↓ 방향 | 그래프 확인 ✓ |
| 23.2 | RLHF vs DPO | 비교 정리 ✓ |
| 23.5 | 정렬의 부작용 | 정리 ✓ |

### DPO의 핵심

```python
# 정책이 참조 대비 얼마나 올랐는가
chosen_reward   = policy_chosen_logps   - ref_chosen_logps
rejected_reward = policy_rejected_logps - ref_rejected_logps

# 격차를 벌리도록 학습
loss = -F.logsigmoid(beta * (chosen_reward - rejected_reward))
```

**세 줄이 전부다.** RLHF의 복잡한 강화학습 루프가 이것으로 대체된다.

### 기억할 것

| 항목 | 요점 |
|---|---|
| 학습 대상 | 절대 확률이 아니라 **격차** |
| 참조 모델 | 원본에서 벗어나는 것을 억제 |
| 메모리 | 모델 두 개 — LoRA와 함께 쓰면 절약 |
| 학습률 | SFT보다 훨씬 작게 (5e-6 수준) |
| 초기 손실 | 항상 ln2 = 0.693 |
| beta | 0.1~0.5, 크면 공격적 |
| `logsigmoid` | 수치 안정성 |
| 응답 길이 | 길이 차이가 크면 보정 필요 |

### 학습 파이프라인 전체

| 단계 | 장 | 가르치는 것 |
|---|---|---|
| 사전학습 | (23장에서 결과만 사용) | 언어 자체 |
| **SFT** | **24번** | 지시를 따르는 법 |
| **정렬** | **26번** | 어떤 답이 더 나은가 |
| 효율화 | 25번 | 적은 자원으로 위 과정 수행 |

### 다음 장

**35. Reasoning — 단계적 사고** — 이론편 23.6절.
"단계적으로 생각하라"는 지시 하나가 성능을 바꾸는 현상과,
그것이 왜 작동하는지를 다룬다.